# Python classes — part 2

**Week 2 · Notebook 2 of 3 · Initialisation, callable objects, and subclassing**

Complete **`Week_02_intro_to_pytorch.ipynb`** first. Run this notebook from the top in **Google Colab**. It imports what it needs, so it does not require variables from another notebook. No GPU is needed.

In the Week 1 lab, we introduced Python classes and saw how objects can store data and provide methods. We now develop these ideas a little further using PyTorch tensors. In particular, we will see how an object can create and store its own state, how an object can be called like a function, and how one class can build on another. These ideas will be useful later when we implement neural networks in PyTorch.

## Computing an instance's initial state

In Week 1, `__init__` stored an input in an attribute. It can also **compute** the initial state.

`RandomTranslate` below draws a random vector when an instance is created and stores it as `self.translation`. Its `translate` method then adds that **same stored vector** to each input.

The parameter `dim` selects the number of entries: `2` for vectors of length 2, `3` for vectors of length 3, and so on. A length-3 vector is represented by a **1-dimensional tensor of shape `(3,)`**, not by a 3-dimensional array.


In [1]:
import torch as t


class RandomTranslate:
    def __init__(self, dim):
        # Draw once during initialisation and store the result on this instance.
        self.translation = t.randn(dim)

    def translate(self, vec):
        # `vec` is the input to this call; self.translation is the stored state.
        # For these examples, use a vector of the same length as the translation.
        return vec + self.translation


In [2]:
# Create an instance for length-2 vectors; dim receives 2 in __init__.
random_translate_2Da = RandomTranslate(2)

# Adding the translation to a zero vector shows the translation itself.
random_translate_2Da.translate(t.zeros(2))


tensor([ 0.0374, -0.8356])

In [3]:
# A second instance draws its own translation, generally different from the first.
# Creating it does not change the state of random_translate_2Da.
random_translate_2Db = RandomTranslate(2)
random_translate_2Db.translate(t.zeros(2))


tensor([-0.9188, -0.1810])

In [4]:
# The same class works for length-3 vectors: dim now receives 3.
random_translate_3D = RandomTranslate(3)
random_translate_3D.translate(t.zeros(3))


tensor([-1.0126, -1.3041,  1.8916])

In [5]:
# Neither call below redraws the translation: both read the stored attribute.
print(random_translate_2Da.translate(t.zeros(2)))
a = t.ones(2)
print(random_translate_2Da.translate(a) - a)
# Both calculations recover the same translation, up to floating-point rounding.
# __init__ ran when this instance was created; translate runs on each call.


tensor([ 0.0374, -0.8356])
tensor([ 0.0374, -0.8356])


The important point is that different objects can store different state, and that this state can persist between calls. We will use this idea again later in the course.


## Making objects behave like functions

We now have an object that can create and keep its own state. We next want to make an object itself behave like a function when we give it an input.

So far, we have called a named method:

```python
random_translate_2Da.translate(input_vector)
```

We can also define what it means to call an **instance itself**, using syntax such as:

```python
random_translate_2D(input_vector)
```

The special method **`__call__`** provides this behaviour. Like `__init__`, its name has two underscores on each side. In the next class, we replace the method name `translate` with `__call__`. The calculation is unchanged.


In [6]:
class RandomTranslateFunc:
    def __init__(self, dim):
        self.translation = t.randn(dim)

    def __call__(self, vec):
        # Calling an instance supplies that instance as self and the input as vec.
        return vec + self.translation


In [7]:
# Calling the CLASS creates an instance and runs its __init__ with dim=2.
random_translate_2D = RandomTranslateFunc(2)

# Calling that INSTANCE runs its __call__ with vec=t.zeros(2).
# This does not create another instance or rerun __init__.
random_translate_2D(t.zeros(2))


tensor([ 0.0478, -0.4297])

This combination of stored state and function-like behaviour will be useful later when we build neural networks.


## Subclassing [Non examinable]

There is one final idea about Python classes that will be useful later: one class can build on another.

A **subclass** builds on another class, called its **superclass** or **base class**. It can use methods from that class and define methods of its own.

The details in this section remain **non-examinable**. The examples below explain the inheritance syntax and what `super().__init__()` does.


In [8]:
class MySuperClass:
    def __init__(self):
        print("Calling MySuperClass __init__")
        self.a = 1  # This attribute is set only if this initialiser runs.

    def superclass_method1(self, x):
        return 2 * x  # Does not read any instance attributes.

    def superclass_method2(self, x):
        return x + self.a  # Needs the instance to have an attribute a.


In [9]:
# The class name in parentheses specifies the superclass to inherit from.
class MySubClass(MySuperClass):
    def __init__(self):
        # This class defines its OWN __init__, which takes precedence.
        print("Calling MySubClass __init__")
        self.b = 3
        self.c = 4

    def subclass_method1(self, x):
        return x + self.b

    def subclass_method2(self, x):
        return x + self.c


In [10]:
# MySubClass uses its own methods and inherits the other methods of MySuperClass.
# Its own __init__ overrides the inherited one: the parent initialiser is NOT
# automatically run as well. This instance therefore has b and c, but not a.
myobj = MySubClass()
myobj.subclass_method1(3)  # 3 + myobj.b = 6.


Calling MySubClass __init__


6

In [11]:
# The inherited method is available and works: it only needs the argument x.
myobj.superclass_method1(3)  # 2 * 3 = 6.


6

In [12]:
# The other inherited method is available too, but it tries to read self.a.
# That attribute was not set, because MySuperClass.__init__ did not run.
# Uncomment the call to see the intentional error, then comment it out again.
# myobj.superclass_method2(4)  # AttributeError: no attribute 'a'.


In [13]:
class MySubClass2(MySuperClass):
    def __init__(self):
        # In this single-inheritance example, run MySuperClass's initialiser
        # on this SAME instance, setting self.a before continuing below.
        super().__init__()
        print("Calling MySubClass2 __init__")
        self.c = 3
        self.d = 4

    def subclass_method1(self, x):
        return x + self.c

    def subclass_method2(self, x):
        return x + self.d


In [14]:
# Both initialisers now run, setting a, c, and d on this instance.
myobj = MySubClass2()
print(myobj.a)                  # 1, set by the superclass's initialiser.
myobj.superclass_method1(3)     # Still returns 6.


Calling MySuperClass __init__
Calling MySubClass2 __init__
1


6

In [15]:
# The inherited method that failed above now works, because myobj.a exists.
myobj.superclass_method2(4)  # 4 + 1 = 5.


5